In [ ]:
import time

import pandas as pd
import requests

BASE_URL = "https://gutendex.com/books"
OUTPUT_CSV = "proper_genre_gutenberg_10k.csv"
PARTIAL_CSV = "gutenberg_partial.csv"
SAVE_EVERY_N_PAGES = 25


def determine_proper_genre(subjects_str, bookshelves_str):
    combined = (subjects_str + " " + bookshelves_str).lower()
    if "science fiction" in combined or "sci-fi" in combined:
        return "Science Fiction"
    elif "horror" in combined or "ghost stories" in combined or "gothic" in combined:
        return "Horror"
    elif "fantasy" in combined or "fairy tales" in combined:
        return "Fantasy"
    elif "adventure" in combined or "picaresque" in combined:
        return "Adventure"
    elif "philosophy" in combined or "ethics" in combined:
        return "Philosophy"
    elif "poetry" in combined or "poems" in combined:
        return "Poetry"
    elif "drama" in combined or "plays" in combined or "tragedy" in combined:
        return "Drama"
    elif "history" in combined or "historical" in combined:
        return "History"
    elif "biograph" in combined or "memoir" in combined or "autobiograph" in combined:
        return "Biography & Memoir"
    elif "detective" in combined or "mystery" in combined or "crime" in combined:
        return "Mystery & Crime"
    elif "juvenile" in combined or "children" in combined:
        return "Children's Literature"
    elif "religion" in combined or "theology" in combined or "christianity" in combined:
        return "Religion & Spirituality"
    elif "science" in combined or "astronomy" in combined or "biology" in combined:
        return "Science & Technology"
    elif "politics" in combined or "political" in combined:
        return "Politics & Government"
    elif "essays" in combined or "letters" in combined:
        return "Essays & Letters"
    elif "fiction" in combined or "novel" in combined or "stories" in combined:
        return "General Fiction"
    else:
        return "General Non-Fiction / Other"


def fetch_page(session, url, max_retries=5):
    """Fetch one page. Only network/HTTP/JSON problems are retried."""
    for attempt in range(1, max_retries + 1):
        try:
            response = session.get(url, timeout=30)
            if response.status_code == 200:
                return response.json()
            print(f"\n⚠️ Status {response.status_code}, retry {attempt}/{max_retries}...")
        except (requests.exceptions.RequestException, ValueError) as e:
            print(f"\n⚠️ Network issue ({type(e).__name__}): {e} | retry {attempt}/{max_retries}")
        time.sleep(5 * attempt)  # increasing backoff
    return None


def build_record(book, genre):
    authors = book.get("authors") or []
    author_name = authors[0].get("name", "Unknown Author") if authors else "Unknown Author"

    summaries = book.get("summaries") or []
    formats = book.get("formats") or {}

    return {
        "book_id": book.get("id"),
        "title": book.get("title", "Unknown Title"),
        "author": author_name,
        "summary": " ".join(summaries),
        "subjects": ", ".join(book.get("subjects") or []),
        "bookshelves": ", ".join(book.get("bookshelves") or []),
        "cover_image_url": formats.get("image/jpeg", ""),
        "download_count": book.get("download_count", 0),
        "genre": genre,
    }


def save_partial(fiction_list, other_list):
    if fiction_list or other_list:
        pd.DataFrame(fiction_list + other_list).to_csv(
            PARTIAL_CSV, index=False, encoding="utf-8"
        )


def extract_curated_gutenberg_dataset(target_fiction=3000, target_other=7000):
    total_target = target_fiction + target_other
    url = BASE_URL

    fiction_list = []
    other_list = []
    seen_ids = set()

    print("🚀 Initializing extraction loop...")
    print(f"🎯 Targets: {target_fiction} Fiction books | {target_other} Other genres.")
    print("-" * 70)

    session = requests.Session()
    page_count = 1
    fiction_capped = False
    other_capped = False

    while url and (len(fiction_list) < target_fiction or len(other_list) < target_other):
        print(f"📖 Page {page_count:03d} -> Fetching... ", end="", flush=True)

        data = fetch_page(session, url)
        if data is None:
            print("🛑 Giving up after repeated failures.")
            break

        results = data.get("results", [])
        if not results:
            print("\n🛑 No further book results from the API.")
            break

        page_fiction_count = 0
        page_other_count = 0

        for book in results:
            if book.get("media_type") != "Text":
                continue

            book_id = book.get("id")
            if book_id in seen_ids:
                continue
            seen_ids.add(book_id)

            subjects_str = ", ".join(book.get("subjects") or [])
            bookshelves_str = ", ".join(book.get("bookshelves") or [])
            genre = determine_proper_genre(subjects_str, bookshelves_str)

            combined_tags = (subjects_str + " " + bookshelves_str).lower()
            is_fiction = (
                "fiction" in combined_tags
                or "stories" in combined_tags
                or "novel" in combined_tags
            )

            record = build_record(book, genre)

            if is_fiction:
                if len(fiction_list) < target_fiction:
                    fiction_list.append(record)
                    page_fiction_count += 1
            else:
                if len(other_list) < target_other:
                    other_list.append(record)
                    page_other_count += 1

        print(
            f"+{page_fiction_count} Fiction, +{page_other_count} Others | "
            f"Total: {len(fiction_list) + len(other_list)}/{total_target}"
        )

        if len(fiction_list) >= target_fiction and not fiction_capped:
            print(f"🔒 [MILESTONE] Fiction pool ({target_fiction}) filled!")
            fiction_capped = True

        if len(other_list) >= target_other and not other_capped:
            print(f"🔒 [MILESTONE] Other genres pool ({target_other}) filled!")
            other_capped = True

        if page_count % SAVE_EVERY_N_PAGES == 0:
            save_partial(fiction_list, other_list)
            print(f"💾 Partial progress saved -> '{PARTIAL_CSV}'")

        url = data.get("next")
        page_count += 1
        time.sleep(1)  # courteous cooldown

    print("-" * 70)
    print("Writing records to CSV, ranked by download count...")

    final_dataset = fiction_list + other_list
    if not final_dataset:
        print("⚠️ No records collected, nothing to save.")
        return pd.DataFrame()

    df = pd.DataFrame(final_dataset)
    df = df.sort_values(by="download_count", ascending=False)
    df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8")

    print(f"🎉 SUCCESS: {len(df)} books saved -> '{OUTPUT_CSV}'")
    print(f"   Fiction: {len(fiction_list)} | Other: {len(other_list)}")
    return df


if __name__ == "__main__":
    start_time = time.time()
    df_books = extract_curated_gutenberg_dataset(target_fiction=3000, target_other=7000)
    print(f"⏱️ Total run-time: {round((time.time() - start_time) / 60, 2)} minutes.")

🚀 Initializing extraction loop...
🎯 Targets: 3000 Fiction books | 7000 Other genres.
----------------------------------------------------------------------
📖 Page 001 -> Fetching... +23 Fiction, +9 Others | Total: 32/10000
📖 Page 002 -> Fetching... +21 Fiction, +10 Others | Total: 63/10000
📖 Page 003 -> Fetching... +23 Fiction, +9 Others | Total: 95/10000
📖 Page 004 -> Fetching... +13 Fiction, +19 Others | Total: 127/10000
📖 Page 005 -> Fetching... +18 Fiction, +14 Others | Total: 159/10000
📖 Page 006 -> Fetching... +15 Fiction, +17 Others | Total: 191/10000
📖 Page 007 -> Fetching... +16 Fiction, +16 Others | Total: 223/10000
📖 Page 008 -> Fetching... +24 Fiction, +8 Others | Total: 255/10000
📖 Page 009 -> Fetching... +22 Fiction, +8 Others | Total: 285/10000
📖 Page 010 -> Fetching... +20 Fiction, +12 Others | Total: 317/10000
📖 Page 011 -> Fetching... +16 Fiction, +15 Others | Total: 348/10000
📖 Page 012 -> Fetching... +21 Fiction, +11 Others | Total: 380/10000
📖 Page 013 -> Fetching.